# 7. Reproducing a published estimate: the Hubbard model

Babbush et al. [[arXiv:1805.03662](https://arxiv.org/abs/1805.03662), Table IV] estimated the T count of the ground-state energy of the 2D Hubbard model, from $6\times6$ to $20\times20$ sites, by qubitized phase estimation. Can the table be rebuilt from the pieces of notebooks 03 and 04, and what does the model's structure save?

In [1]:
import numpy as np
from openfermion import fermi_hubbard, jordan_wigner

import qcost
from qcost.exact import from_openfermion
from qcost.hubbard import TABLE_IV, hubbard_estimate, hubbard_lambda, hubbard_select_toffolis
from qcost.walk import walk_calls

## The table, rebuilt

The model is $H = -t\sum_{\langle pq\rangle,\sigma} a^\dagger_{p\sigma}a_{q\sigma} + u\sum_p n_{p\uparrow}n_{p\downarrow}$ on an $M\times M$ periodic square lattice, $N = 2M^2$ spin orbitals, at $u = 4t$ and energy error $\Delta E = t/100$. Three numbers carry the whole estimate:

* $\lambda = 2Nt + Nu/2$ [Eq. 60];
* walk calls $\sqrt2\,\pi\lambda/\Delta E$ [Eq. 26], with $\Delta E$ a root-mean-square error;
* $10N$ T gates per call, all in SELECT [Sec. V A].

In [2]:
print(f"{'lattice':>8} {'N':>4} {'T (ours)':>10} {'T (paper)':>10}"
      f" {'ancilla':>8} {'paper':>6} {'logical':>8} {'paper':>6}")
for name, row in TABLE_IV.items():
    e = hubbard_estimate(row["N"], t=1.0, u=4.0, dE=0.01)
    print(f"{name:>8} {row['N']:4d} {e['T']:10.2g} {row['T']:10.2g} {e['ancilla']:8d} {row['ancilla']:6d}"
          f" {e['logical']:8d} {row['logical']:6d}")

 lattice    N   T (ours)  T (paper)  ancilla  paper  logical  paper
   6 x 6   72    9.2e+07    9.3e+07       33     33      105    105
   8 x 8  128    2.9e+08    2.9e+08       33     33      161    161
 10 x 10  200    7.1e+08    7.1e+08       36     36      236    236
 20 x 20  800    1.1e+10    1.2e+10       42     42      842    842


Every qubit count matches, and every T count to the two figures printed, except the smallest and largest lattices, which come out 1% and 5% low ($9.2$ vs $9.3\times10^7$, $1.14$ vs $1.2\times10^{10}$). The paper's $10N + O(\log N)$ per call and its PREPARE, $O(\log(N/\epsilon))$, both dropped here, are the likely source.

## Checking the inputs

**The one-norm.** Build the Hamiltonian with OpenFermion, map it to qubits, and add up the coefficients. Eq. 60 includes the constant term, $Nu/8$, which a phase estimation does not pay for; without it, $\lambda$ is 12.5% smaller at $u = 4t$:

In [3]:
for M in (3, 4):
    op = jordan_wigner(fermi_hubbard(M, M, 1.0, 4.0, periodic=True))
    N = 2 * M * M
    with_id = sum(abs(c) for c in op.terms.values())
    P = from_openfermion(op, N)
    print(f"{M}x{M}: Eq. 60 gives {hubbard_lambda(N):.1f};  Jordan-Wigner sum {with_id:.1f} with the constant,"
          f" {P.one_norm:.1f} without;  {P.L} Pauli terms")

3x3: Eq. 60 gives 72.0;  Jordan-Wigner sum 72.0 with the constant, 63.0 without;  99 Pauli terms
4x4: Eq. 60 gives 128.0;  Jordan-Wigner sum 128.0 with the constant, 112.0 without;  176 Pauli terms


**The SELECT.** It is three unary iterations: two "selected Majorana" operators over $N$ items and one indexed $Z$ over $N/2$. Notebook 03 counts a unary iteration over $L$ items at $L - 1$ Toffolis, i.e. $4L - 4$ T gates, so $4N + 4N + 2N = 10N$ T:

In [4]:
for name, row in TABLE_IV.items():
    N = row["N"]
    tof = hubbard_select_toffolis(N)
    print(f"{name:>8}: our unary iterations {tof:5d} Toffolis = {4 * tof:5d} T;  paper 10N = {10 * N}")

   6 x 6: our unary iterations   177 Toffolis =   708 T;  paper 10N = 720
   8 x 8: our unary iterations   317 Toffolis =  1268 T;  paper 10N = 1280
 10 x 10: our unary iterations   497 Toffolis =  1988 T;  paper 10N = 2000
 20 x 20: our unary iterations  1997 Toffolis =  7988 T;  paper 10N = 8000


**The precision convention.** Babbush et al. define $\Delta E$ as a root-mean-square error and need $\sqrt2\pi\lambda/\Delta E$ calls; Lee et al. [[arXiv:2011.03494](https://arxiv.org/abs/2011.03494), Eq. 45], used everywhere else in this repository, need $\pi\lambda/(2\epsilon)$. The ratio is $2\sqrt2 \approx 2.8$. Reading a published number means reading its definition of the error first:

In [5]:
N = 128
print(f"8x8: RMS convention {np.sqrt(2) * np.pi * hubbard_lambda(N) / 0.01:.3g} calls;"
      f" Lee et al. convention {walk_calls(hubbard_lambda(N), 0.01):.3g} calls")

8x8: RMS convention 2.27e+05 calls; Lee et al. convention 8.04e+04 calls


## What the structure saves

The Hubbard model has only three distinct coefficients and is translation invariant. So SELECT needs no data, and PREPARE is two rotations and uniform superpositions: no QROM, no alias sampling [Sec. V B]. Run the generic pipeline of notebook 04 on the same Hamiltonian, which treats it as an arbitrary list of Pauli terms:

In [6]:
M = 8
op = jordan_wigner(fermi_hubbard(M, M, 1.0, 4.0, periodic=True))
P = from_openfermion(op, 2 * M * M)
generic = qcost.estimate(P, eps=0.01)
N = 2 * M * M
specialized = hubbard_select_toffolis(N)
print(f"generic: {P.L} terms, {generic['Toffolis per walk step']} Toffolis per step"
      f" (SELECT {generic['  step: SELECT']}, PREPARE {generic['  step: PREPARE']} x 2)")
print(f"specialized: about {specialized} Toffolis per step (SELECT only)")
print(f"ratio {generic['Toffolis per walk step'] / specialized:.1f}")

generic: 704 terms, 2224 Toffolis per step (SELECT 703, PREPARE 747 x 2)
specialized: about 317 Toffolis per step (SELECT only)
ratio 7.0


The generic walk is several times dearer per step: it loads a weight for every one of the Pauli terms, though there are only three distinct values, and its SELECT iterates over all the terms instead of over sites. Structure is worth more than any constant-factor trick in the generic circuit.

## Against Trotter

Kivlichan et al. [[arXiv:1902.10673](https://arxiv.org/abs/1902.10673), Table 2] costed the same $8\times8$ model, same $\Delta E = t/100$, with second-order Trotter steps: $4.6\times10^7$ Toffolis plus $5.0\times10^9$ T gates on 132 logical qubits, against $2.9\times10^8$ T for qubitization above. They note that qubitization wins for a fixed (intensive) error, and Trotter for an error allowed to grow with the system [their Sec. 4]; their Trotter step counts rest on numerically computed error norms, which is why they are quoted here rather than rebuilt.

## Exercises

1. Redo Table IV at $u = 8t$. Which lattice crosses $10^{10}$ T gates first?
2. Using the constant-free $\lambda$ from the check above, by how much would each T count drop?
3. Turn the T counts into physical qubits and hours with `qcost.surface_code.footprint` (divide T by 4 for Toffolis). How long does the $20\times20$ lattice take with one factory?